# CrimeScope ML — 03. Panel, ACS, Crime-Type Split & Features (Lakehouse)

**Description:** Builds the dense tract×month panel with labels, fetches ACS data,
splits crimes into violent/property categories, computes per-capita rates, engineers
interaction features, spatial context, and writes the full feature matrix as Delta.

**Improvements over v1:**
- Violent vs property crime type split with separate lag/rolling features
- Per-capita crime rates and interaction features for SHAP diversity
- Community-area spatial context (neighbor average)
- Year-over-year and trend acceleration features

**Tables written:**
- `tract_crime_monthly_panel_labeled` (with violent/property counts)
- `tract_acs_population`
- `tract_crime_monthly_panel_with_acs`
- `tract_crime_features`

**Prerequisites:** `02` (crime monthly by tract + tract boundaries + crimes_with_tract).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")
print("Catalog:", spark.sql("SELECT current_catalog()").first()[0],
      "| Schema:", spark.sql("SELECT current_schema()").first()[0])

---
## 1. Crime-type classification and monthly counts by type

In [ ]:
VIOLENT_TYPES = [
    "HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
    "CRIMINAL SEXUAL ASSAULT", "KIDNAPPING", "HUMAN TRAFFICKING",
    "CRIM SEXUAL ASSAULT",
]
PROPERTY_TYPES = [
    "THEFT", "BURGLARY", "MOTOR VEHICLE THEFT", "ARSON",
    "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
]

cwt = spark.table("varanasi.default.chicago_crimes_with_tract")
cwt = cwt.filter(F.col("tract_geoid").isNotNull())
cwt = cwt.withColumn("month_start", F.date_trunc("month", F.to_timestamp("date")))

cwt = cwt.withColumn(
    "crime_category",
    F.when(F.upper(F.col("primary_type")).isin(VIOLENT_TYPES), F.lit("violent"))
     .when(F.upper(F.col("primary_type")).isin(PROPERTY_TYPES), F.lit("property"))
     .otherwise(F.lit("other"))
)

type_monthly = (
    cwt.groupBy("tract_geoid", "month_start")
    .agg(
        F.count("*").alias("incident_count"),
        F.sum(F.when(F.col("crime_category") == "violent", 1).otherwise(0)).alias("violent_count"),
        F.sum(F.when(F.col("crime_category") == "property", 1).otherwise(0)).alias("property_count"),
        F.sum(F.when(F.col("crime_category") == "other", 1).otherwise(0)).alias("other_count"),
    )
)

# Also compute community-area level counts for spatial context
ca_monthly = (
    cwt.filter(F.col("community_area").isNotNull())
    .groupBy("community_area", "month_start")
    .agg(F.count("*").alias("ca_incident_count"))
)

# Map tract to community area (most common CA for each tract)
tract_ca = (
    cwt.filter(F.col("community_area").isNotNull())
    .groupBy("tract_geoid", "community_area")
    .agg(F.count("*").alias("n"))
)
w_ca = Window.partitionBy("tract_geoid").orderBy(F.desc("n"))
tract_ca = tract_ca.withColumn("rk", F.row_number().over(w_ca)).filter("rk = 1").drop("n", "rk")

print(f"Crime type distribution:")
display(cwt.groupBy("crime_category").count().orderBy(F.desc("count")))
print(f"Tract-CA mapping: {tract_ca.count()} tracts")

---
## 2. Dense panel + labels (with crime type counts)

In [ ]:
tracts = spark.table("varanasi.default.cook_tract_boundaries").select("tract_geoid")
months = type_monthly.select("month_start").distinct()

panel = tracts.crossJoin(months)
panel = panel.join(type_monthly, ["tract_geoid", "month_start"], "left")
panel = panel.fillna(0, subset=["incident_count", "violent_count", "property_count", "other_count"])

# Join community area mapping
panel = panel.join(tract_ca, "tract_geoid", "left")

# Trailing 12-month rolling incident counts
w12 = Window.partitionBy("tract_geoid").orderBy("month_start").rowsBetween(-11, 0)
panel = panel.withColumn("y_incidents_12m", F.sum("incident_count").over(w12))
panel = panel.withColumn("y_violent_12m", F.sum("violent_count").over(w12))
panel = panel.withColumn("y_property_12m", F.sum("property_count").over(w12))

# Next-30-day proxy: next month's counts
w_future = Window.partitionBy("tract_geoid").orderBy("month_start")
panel = panel.withColumn("y_next_30d_count", F.lead("incident_count", 1).over(w_future))
panel = panel.withColumn("y_next_30d_violent", F.lead("violent_count", 1).over(w_future))
panel = panel.withColumn("y_next_30d_property", F.lead("property_count", 1).over(w_future))

out = panel.filter(F.col("y_next_30d_count").isNotNull())

MATURITY_BUFFER_MONTHS = 3
panel_max = out.select(F.max("month_start")).first()[0]
maturity_cutoff = out.select(F.add_months(F.lit(panel_max), -MATURITY_BUFFER_MONTHS)).first()[0]
out = out.filter(F.col("month_start") <= maturity_cutoff)
print(f"Maturity buffer: dropped months after {maturity_cutoff}")

PANEL_TABLE = "varanasi.default.tract_crime_monthly_panel_labeled"
spark.sql(f"DROP TABLE IF EXISTS {PANEL_TABLE}")
out.write.saveAsTable(PANEL_TABLE)
spark.sql(f"OPTIMIZE {PANEL_TABLE} ZORDER BY (tract_geoid, month_start)")
spark.sql(f"COMMENT ON TABLE {PANEL_TABLE} IS 'Dense panel with violent/property split and next-30d labels'")

n = spark.table(PANEL_TABLE).count()
print(f"Panel rows: {n:,} (Delta, Z-ordered)")
display(spark.table(PANEL_TABLE).limit(10))

---
## 3. ACS population + income (Census API → Delta)

In [ ]:
import json
import urllib.parse
import urllib.request

import pandas as pd

base = "https://api.census.gov/data/2022/acs/acs5"
params = urllib.parse.urlencode([
    ("get", "B01003_001E,B19013_001E,B17001_002E,B25001_001E,NAME"),
    ("for", "tract:*"),
    ("in", "state:17"),
    ("in", "county:031"),
])
url = f"{base}?{params}"

with urllib.request.urlopen(url, timeout=60) as resp:
    rows = json.loads(resp.read().decode())

header, data = rows[0], rows[1:]
pdf = pd.DataFrame(data, columns=header)
pdf["tract_geoid"] = pdf["state"] + pdf["county"] + pdf["tract"]
pdf["total_pop_acs"] = pd.to_numeric(pdf["B01003_001E"], errors="coerce")
pdf["median_hh_income_acs"] = pd.to_numeric(pdf["B19013_001E"], errors="coerce")
pdf["poverty_count_acs"] = pd.to_numeric(pdf["B17001_002E"], errors="coerce")
pdf["housing_units_acs"] = pd.to_numeric(pdf["B25001_001E"], errors="coerce")
pdf["acs_year"] = 2022

pdf["poverty_rate_acs"] = (pdf["poverty_count_acs"] / pdf["total_pop_acs"]).where(pdf["total_pop_acs"] > 0)

acs_cols = ["tract_geoid", "total_pop_acs", "median_hh_income_acs",
            "poverty_count_acs", "poverty_rate_acs", "housing_units_acs", "acs_year"]
sdf = spark.createDataFrame(pdf[acs_cols])

boundaries = spark.table("varanasi.default.cook_tract_boundaries").select("tract_geoid")
sdf = sdf.join(boundaries, "tract_geoid", "inner")

ACS_TABLE = "varanasi.default.tract_acs_population"
spark.sql(f"DROP TABLE IF EXISTS {ACS_TABLE}")
sdf.write.saveAsTable(ACS_TABLE)
spark.sql(f"OPTIMIZE {ACS_TABLE} ZORDER BY (tract_geoid)")
spark.sql(f"COMMENT ON TABLE {ACS_TABLE} IS 'ACS 2022 5-year: population, income, poverty, housing for Cook County tracts'")

print(f"ACS tracts: {spark.table(ACS_TABLE).count()} (Delta, Z-ordered)")
display(spark.table(ACS_TABLE).limit(5))

---
## 4. Join panel + ACS, compute per-1k rates

In [ ]:
panel = spark.table(PANEL_TABLE)
acs = spark.table(ACS_TABLE)

joined = panel.join(acs, "tract_geoid", "left")

for col_name, src_col in [("y_rate_12m_per_1k", "y_incidents_12m"),
                           ("y_next_30d_per_1k", "y_next_30d_count")]:
    joined = joined.withColumn(
        col_name,
        F.when((F.col("total_pop_acs").isNotNull()) & (F.col("total_pop_acs") > 0),
               F.col(src_col) / (F.col("total_pop_acs") / 1000)).otherwise(F.lit(None)),
    )

PANEL_ACS_TABLE = "varanasi.default.tract_crime_monthly_panel_with_acs"
spark.sql(f"DROP TABLE IF EXISTS {PANEL_ACS_TABLE}")
joined.write.saveAsTable(PANEL_ACS_TABLE)
spark.sql(f"OPTIMIZE {PANEL_ACS_TABLE} ZORDER BY (tract_geoid, month_start)")

print(f"Panel with ACS: {spark.table(PANEL_ACS_TABLE).count():,} (Delta, Z-ordered)")

---
## 5. Feature engineering

In [ ]:
import math
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table(PANEL_ACS_TABLE)
w = Window.partitionBy("tract_geoid").orderBy("month_start")

### 5a. Overall lag features

In [ ]:
for lag in [1, 2, 3, 6, 12]:
    df = df.withColumn(f"lag_{lag}m_count", F.lag("incident_count", lag).over(w))

for period in [3, 6, 12]:
    w_roll = Window.partitionBy("tract_geoid").orderBy("month_start").rowsBetween(-period, -1)
    df = df.withColumn(f"rolling_mean_{period}m", F.avg("incident_count").over(w_roll))
    if period >= 6:
        df = df.withColumn(f"rolling_std_{period}m", F.stddev("incident_count").over(w_roll))
        df = df.withColumn(f"rolling_max_{period}m", F.max("incident_count").over(w_roll))
        df = df.withColumn(f"rolling_min_{period}m", F.min("incident_count").over(w_roll))

df = df.withColumn("mom_change", F.col("incident_count") - F.lag("incident_count", 1).over(w))
print("Overall lag + rolling features added")

### 5b. Violent and property crime lag features

In [ ]:
for crime_type in ["violent", "property"]:
    src_col = f"{crime_type}_count"
    for lag in [1, 3, 6]:
        df = df.withColumn(f"{crime_type}_lag_{lag}m", F.lag(src_col, lag).over(w))
    for period in [3, 6, 12]:
        w_roll = Window.partitionBy("tract_geoid").orderBy("month_start").rowsBetween(-period, -1)
        df = df.withColumn(f"{crime_type}_rolling_{period}m", F.avg(src_col).over(w_roll))

# Violent ratio (proportion of crimes that are violent)
df = df.withColumn(
    "violent_ratio",
    F.when(F.col("incident_count") > 0, F.col("violent_count") / F.col("incident_count")).otherwise(0.0)
)
# Rolling violent ratio
w6 = Window.partitionBy("tract_geoid").orderBy("month_start").rowsBetween(-6, -1)
df = df.withColumn("violent_ratio_6m", F.avg("violent_ratio").over(w6))

print("Violent + property crime features added")

### 5c. Calendar / seasonality

In [ ]:
df = df.withColumn("month_of_year", F.month("month_start"))
df = df.withColumn("month_sin", F.sin(2 * math.pi * F.col("month_of_year") / 12))
df = df.withColumn("month_cos", F.cos(2 * math.pi * F.col("month_of_year") / 12))
df = df.withColumn("year", F.year("month_start"))

# Same month last year
df = df.withColumn(
    "same_month_last_year",
    F.lag("incident_count", 12).over(Window.partitionBy("tract_geoid").orderBy("month_start"))
)
# Year-over-year change
df = df.withColumn(
    "yoy_change",
    F.when(F.col("same_month_last_year").isNotNull(),
           F.col("incident_count") - F.col("same_month_last_year")).otherwise(F.lit(None))
)
print("Calendar + YoY features added")

### 5d. ACS contextual + interaction features

In [ ]:
df = df.withColumn(
    "log_pop",
    F.when(F.col("total_pop_acs") > 0, F.log(F.col("total_pop_acs"))).otherwise(0.0),
)
df = df.withColumn(
    "log_income",
    F.when(F.col("median_hh_income_acs") > 0, F.log(F.col("median_hh_income_acs"))).otherwise(0.0),
)

# Crime rate per 1,000 residents (using rolling mean for stability)
df = df.withColumn(
    "crime_rate_per_1k",
    F.when((F.col("total_pop_acs") > 0) & (F.col("rolling_mean_12m").isNotNull()),
           F.col("rolling_mean_12m") / (F.col("total_pop_acs") / 1000.0)).otherwise(0.0)
)

# Poverty × crime interaction
df = df.withColumn(
    "poverty_x_crime",
    F.coalesce(F.col("poverty_rate_acs"), F.lit(0.0)) * F.coalesce(F.col("rolling_mean_12m"), F.lit(0.0))
)

# Income-crime ratio (higher income with high crime is unusual)
df = df.withColumn(
    "income_crime_ratio",
    F.when(F.col("rolling_mean_12m") > 0,
           F.coalesce(F.col("median_hh_income_acs"), F.lit(0.0)) / (F.col("rolling_mean_12m") * 1000.0))
     .otherwise(F.lit(0.0))
)

# Population per housing unit (density proxy)
df = df.withColumn(
    "pop_per_housing_unit",
    F.when((F.col("housing_units_acs").isNotNull()) & (F.col("housing_units_acs") > 0),
           F.col("total_pop_acs") / F.col("housing_units_acs")).otherwise(F.lit(0.0))
)

print("ACS contextual + interaction features added")

### 5e. City-wide and community-area spatial features

In [ ]:
# City-wide totals
city_total = df.groupBy("month_start").agg(
    F.sum("incident_count").alias("city_month_total"),
    F.avg("incident_count").alias("city_month_avg"),
)
df = df.join(city_total, "month_start", "left")

df = df.withColumn(
    "tract_share_of_city",
    F.when(F.col("city_month_total") > 0,
           F.col("incident_count") / F.col("city_month_total")).otherwise(0.0)
)

w_tract = Window.partitionBy("tract_geoid").orderBy("month_start")
df = df.withColumn("city_total_lag1", F.lag("city_month_total", 1).over(w_tract))

# Tract deviation from city average
df = df.withColumn(
    "tract_vs_city_avg",
    F.when(F.col("city_month_avg") > 0,
           F.col("incident_count") / F.col("city_month_avg")).otherwise(1.0)
)

# Community-area average (spatial neighbor proxy)
ca_agg = df.filter(F.col("community_area").isNotNull()).groupBy("community_area", "month_start").agg(
    F.avg("incident_count").alias("ca_avg_crime"),
    F.sum("incident_count").alias("ca_total_crime"),
    F.count("*").alias("ca_n_tracts"),
)
df = df.join(ca_agg, ["community_area", "month_start"], "left")

# Tract relative to its community area
df = df.withColumn(
    "tract_vs_ca_avg",
    F.when((F.col("ca_avg_crime").isNotNull()) & (F.col("ca_avg_crime") > 0),
           F.col("incident_count") / F.col("ca_avg_crime")).otherwise(1.0)
)

print("City-wide + community-area spatial features added")

### 5f. Trend acceleration and volatility

In [ ]:
# Trend acceleration (change in MoM change)
df = df.withColumn(
    "trend_accel",
    F.col("mom_change") - F.lag("mom_change", 1).over(w)
)

# Coefficient of variation (volatility measure)
df = df.withColumn(
    "cv_6m",
    F.when((F.col("rolling_mean_6m").isNotNull()) & (F.col("rolling_mean_6m") > 0),
           F.coalesce(F.col("rolling_std_6m"), F.lit(0.0)) / F.col("rolling_mean_6m"))
     .otherwise(0.0)
)
df = df.withColumn(
    "cv_12m",
    F.when((F.col("rolling_mean_12m").isNotNull()) & (F.col("rolling_mean_12m") > 0),
           F.coalesce(F.col("rolling_std_12m"), F.lit(0.0)) / F.col("rolling_mean_12m"))
     .otherwise(0.0)
)

# 3-month trend direction (is crime rising or falling?)
df = df.withColumn(
    "trend_3m",
    F.when((F.col("rolling_mean_3m").isNotNull()) & (F.col("rolling_mean_6m").isNotNull()) & (F.col("rolling_mean_6m") > 0),
           (F.col("rolling_mean_3m") - F.col("rolling_mean_6m")) / F.col("rolling_mean_6m"))
     .otherwise(0.0)
)

print("Trend + volatility features added")

### 5g. Write feature table to Delta

In [ ]:
FEATURE_COLS = [
    # Overall lag + rolling
    "lag_1m_count", "lag_2m_count", "lag_3m_count", "lag_6m_count", "lag_12m_count",
    "rolling_mean_3m", "rolling_mean_6m", "rolling_mean_12m",
    "rolling_std_6m", "rolling_max_6m", "rolling_min_6m",
    "rolling_std_12m", "rolling_max_12m", "rolling_min_12m",
    "mom_change",
    # Violent crime features
    "violent_lag_1m", "violent_lag_3m", "violent_lag_6m",
    "violent_rolling_3m", "violent_rolling_6m", "violent_rolling_12m",
    "violent_ratio", "violent_ratio_6m",
    # Property crime features
    "property_lag_1m", "property_lag_3m", "property_lag_6m",
    "property_rolling_3m", "property_rolling_6m", "property_rolling_12m",
    # Calendar / seasonality
    "month_of_year", "month_sin", "month_cos", "year",
    "same_month_last_year", "yoy_change",
    # ACS contextual
    "total_pop_acs", "median_hh_income_acs", "poverty_rate_acs", "housing_units_acs",
    "log_pop", "log_income",
    # Interaction / derived
    "crime_rate_per_1k", "poverty_x_crime", "income_crime_ratio", "pop_per_housing_unit",
    # City-wide + spatial context
    "city_month_total", "city_total_lag1", "tract_share_of_city",
    "tract_vs_city_avg", "ca_avg_crime", "tract_vs_ca_avg",
    # Trend + volatility
    "trend_accel", "cv_6m", "cv_12m", "trend_3m",
]

LABEL_COLS = [
    "y_incidents_12m", "y_next_30d_count", "y_rate_12m_per_1k", "y_next_30d_per_1k",
    "y_violent_12m", "y_property_12m", "y_next_30d_violent", "y_next_30d_property",
    "violent_count", "property_count",
]
ID_COLS = ["tract_geoid", "month_start", "community_area"]

out = df.select(*(ID_COLS + FEATURE_COLS + LABEL_COLS))

FEAT_TABLE = "varanasi.default.tract_crime_features"
spark.sql(f"DROP TABLE IF EXISTS {FEAT_TABLE}")
out.write.saveAsTable(FEAT_TABLE)
spark.sql(f"OPTIMIZE {FEAT_TABLE} ZORDER BY (tract_geoid, month_start)")
spark.sql(f"""COMMENT ON TABLE {FEAT_TABLE} IS
  'Wide feature table: {len(FEATURE_COLS)} features + labels for tract-level crime risk modeling'""")

n = spark.table(FEAT_TABLE).count()
print(f"Feature table: {n:,} rows, {len(FEATURE_COLS)} features (Delta, Z-ordered)")
print(f"Features: {FEATURE_COLS}")
display(spark.table(FEAT_TABLE).limit(10))

---
## 6. Feature-table quality assertions

In [ ]:
feat = spark.table(FEAT_TABLE)
n = feat.count()
assert n > 10_000, f"FAIL: feature table has only {n} rows"

lag1_nulls = feat.filter(F.col("lag_1m_count").isNull()).count()
lag1_pct = lag1_nulls / n * 100
print(f"lag_1m_count null: {lag1_nulls:,} ({lag1_pct:.1f}%)")

target_nulls = feat.filter(F.col("y_next_30d_count").isNull()).count()
assert target_nulls == 0, f"FAIL: y_next_30d_count has {target_nulls} nulls"

n_tracts = feat.select("tract_geoid").distinct().count()
n_months = feat.select("month_start").distinct().count()
print(f"\n\u2713 Feature table: {n:,} rows, {n_tracts} tracts, {n_months} months")
print(f"\u2713 {len(FEATURE_COLS)} features including violent/property split")
print(f"\u2713 Violent ratio present: {feat.filter(F.col('violent_ratio') > 0).count():,} rows with violent crimes")
print("\u2705 Feature table ready for notebook 04.")